In [3]:
from google.colab import drive
import os
import sys

drive.mount('/content/drive')
PROJECT_HOME = "/content/drive/MyDrive/IDSC" # Change to project home directory
os.chdir(PROJECT_HOME)

print(f"Current working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/MyDrive/IDSC


In [4]:
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from IPython.display import clear_output
import io

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

"""
── RadioWnet architecture ────────────────────────────────────────────────
from https://github.com/RonLevie/RadioUNet
-
Defined in this .ipynb to load Wnet save-state.
-
To use other models, either create a new block for their architecture
or reference a .py file
──────────────────────────────────────────────────────────────────────────
"""

def convrelu(in_ch, out_ch, kernel, padding, pool):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel, padding=padding),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(pool, stride=pool)
    )

def convreluT(in_ch, out_ch, kernel, padding):
    return nn.Sequential(
        nn.ConvTranspose2d(in_ch, out_ch, kernel, stride=2, padding=padding),
        nn.ReLU(inplace=True)
    )

class RadioWnet(nn.Module):
    def __init__(self, inputs=3, phase="firstU"):
        super().__init__()
        self.inputs = inputs
        self.phase  = phase

        # 1st U-net encoder
        if inputs <= 3:
            self.layer00 = convrelu(inputs, 6,   3, 1, 1)
            self.layer0  = convrelu(6,      40,  5, 2, 2)
        else:
            self.layer00 = convrelu(inputs, 10,  3, 1, 1)
            self.layer0  = convrelu(10,     40,  5, 2, 2)
        self.layer1  = convrelu(40,  50,  5, 2, 2)
        self.layer10 = convrelu(50,  60,  5, 2, 1)
        self.layer2  = convrelu(60,  100, 5, 2, 2)
        self.layer20 = convrelu(100, 100, 3, 1, 1)
        self.layer3  = convrelu(100, 150, 5, 2, 2)
        self.layer4  = convrelu(150, 300, 5, 2, 2)
        self.layer5  = convrelu(300, 500, 5, 2, 2)

        # 1st U-net decoder
        self.conv_up5   = convreluT(500,         300, 4, 1)
        self.conv_up4   = convreluT(300 + 300,   150, 4, 1)
        self.conv_up3   = convreluT(150 + 150,   100, 4, 1)
        self.conv_up20  = convrelu (100 + 100,   100, 3, 1, 1)
        self.conv_up2   = convreluT(100 + 100,    60, 6, 2)
        self.conv_up10  = convrelu ( 60 +  60,    50, 5, 2, 1)
        self.conv_up1   = convreluT( 50 +  50,    40, 6, 2)
        self.conv_up0   = convreluT( 40 +  40,    20, 6, 2)
        skip = 6 if inputs <= 3 else 10
        self.conv_up00  = convrelu(20 + skip + inputs, 20, 5, 2, 1)
        self.conv_up000 = convrelu(20 + inputs,          1, 5, 2, 1)

        # 2nd U-net encoder
        self.Wlayer00 = convrelu(inputs + 1,  20, 3, 1, 1)
        self.Wlayer0  = convrelu( 20,         30, 5, 2, 2)
        self.Wlayer1  = convrelu( 30,         40, 5, 2, 2)
        self.Wlayer10 = convrelu( 40,         50, 5, 2, 1)
        self.Wlayer2  = convrelu( 50,         60, 5, 2, 2)
        self.Wlayer20 = convrelu( 60,         70, 3, 1, 1)
        self.Wlayer3  = convrelu( 70,         90, 5, 2, 2)
        self.Wlayer4  = convrelu( 90,        110, 5, 2, 2)
        self.Wlayer5  = convrelu(110,        150, 5, 2, 2)

        # 2nd U-net decoder
        self.Wconv_up5   = convreluT(150,          110, 4, 1)
        self.Wconv_up4   = convreluT(110 + 110,     90, 4, 1)
        self.Wconv_up3   = convreluT( 90 +  90,     70, 4, 1)
        self.Wconv_up20  = convrelu ( 70 +  70,     60, 3, 1, 1)
        self.Wconv_up2   = convreluT( 60 +  60,     50, 6, 2)
        self.Wconv_up10  = convrelu ( 50 +  50,     40, 5, 2, 1)
        self.Wconv_up1   = convreluT( 40 +  40,     30, 6, 2)
        self.Wconv_up0   = convreluT( 30 +  30,     20, 6, 2)
        self.Wconv_up00  = convrelu (20 + 20 + inputs + 1, 20, 5, 2, 1)
        self.Wconv_up000 = convrelu (20 + inputs + 1,       1, 5, 2, 1)

    def forward(self, x):
        inp = x[:, 0:self.inputs, :, :]
        det1 = self.phase != "firstU"   # detach 1st U-net when training 2nd

        def d1(t): return t.detach() if det1 else t
        l00  = d1(self.layer00(inp));   l0  = d1(self.layer0(l00))
        l1   = d1(self.layer1(l0));     l10 = d1(self.layer10(l1))
        l2   = d1(self.layer2(l10));    l20 = d1(self.layer20(l2))
        l3   = d1(self.layer3(l20));    l4  = d1(self.layer4(l3))
        l5   = d1(self.layer5(l4))

        u = d1(self.conv_up5(l5))
        u = d1(self.conv_up4(torch.cat([u, l4], 1)))
        u = d1(self.conv_up3(torch.cat([u, l3], 1)))
        u = d1(self.conv_up20(torch.cat([u, l20], 1)))
        u = d1(self.conv_up2(torch.cat([u, l2], 1)))
        u = d1(self.conv_up10(torch.cat([u, l10], 1)))
        u = d1(self.conv_up1(torch.cat([u, l1], 1)))
        u = d1(self.conv_up0(torch.cat([u, l0], 1)))
        u = d1(self.conv_up00(torch.cat([u, l00, inp], 1)))
        output1 = d1(self.conv_up000(torch.cat([u, inp], 1)))

        det2 = self.phase != "secondU"
        def d2(t): return t.detach() if det2 else t
        Wi = torch.cat([output1, x], 1).detach()

        w = d2(self.Wlayer00(Wi));    w0  = d2(self.Wlayer0(w))
        w1  = d2(self.Wlayer1(w0));   w10 = d2(self.Wlayer10(w1))
        w2  = d2(self.Wlayer2(w10));  w20 = d2(self.Wlayer20(w2))
        w3  = d2(self.Wlayer3(w20));  w4  = d2(self.Wlayer4(w3))
        w5  = d2(self.Wlayer5(w4))

        wu = d2(self.Wconv_up5(w5))
        wu = d2(self.Wconv_up4(torch.cat([wu, w4],  1)))
        wu = d2(self.Wconv_up3(torch.cat([wu, w3],  1)))
        wu = d2(self.Wconv_up20(torch.cat([wu, w20], 1)))
        wu = d2(self.Wconv_up2(torch.cat([wu, w2],  1)))
        wu = d2(self.Wconv_up10(torch.cat([wu, w10], 1)))
        wu = d2(self.Wconv_up1(torch.cat([wu, w1],  1)))
        wu = d2(self.Wconv_up0(torch.cat([wu, w0],  1)))
        wu = d2(self.Wconv_up00(torch.cat([wu, w, Wi], 1)))
        output2 = d2(self.Wconv_up000(torch.cat([wu, Wi], 1)))

        return [output1, output2]

def build_wnet_input(env, global_heatmap):
    """
    Build the 3-channel tensor RadioWnet expects at inference.
      ch0 : building_mask  (all zeros — the free-space variant has no objects)
      ch1 : tx_origin      (zeros — emitter location is unknown)
      ch2 : sparse path-loss map (cells visited so far, normalised [0,1])
    Returns numpy (3, 256, 256).
    """
    building = env.building_mask.astype(np.float32)
    tx_blank = np.zeros((256, 256), dtype=np.float32)
    return np.stack([building, tx_blank, global_heatmap], axis=0)

def build_goal_map(recon, sigma=40.0):
    """Gaussian goal heatmap centred at the reconstruction argmin (estimated emitter).
    Converts 'find the gradient minimum of a 64x64 map' into 'move toward the visible
    blob' — a far simpler cue for the policy conv-net to learn from."""
    gy, gx = np.unravel_index(np.argmin(recon), recon.shape)
    yg, xg = np.ogrid[:recon.shape[0], :recon.shape[1]]
    return np.exp(-((yg - gy)**2 + (xg - gx)**2) / (2.0 * sigma**2)).astype(np.float32)

print("RadioWnet architecture defined — no external file needed.")


Using device: cuda
RadioWnet architecture defined — no external file needed.


In [5]:
import torch.nn.functional as F

# --- CNN (SpectrumCNN) -----------------------------------------------------
class SpectrumCNN(nn.Module):
    """Encoder-decoder CNN: (B,3,H,W) -> (B,1,H,W) dense path-loss."""
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up3  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec3 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up2  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec2 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up1  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec1 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b  = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(self.up3(b))
        d2 = self.dec2(self.up2(d3))
        d1 = self.dec1(self.up1(d2))
        return self.head(d1)


# --- PartialConvMAE --------------------------------------------------------
class PartialConv2d(nn.Module):
    """Partial convolution: re-normalises by the fraction of valid pixels."""
    def __init__(self, in_ch, out_ch, k, stride=1, padding=0, bias=True):
        super().__init__()
        self.conv       = nn.Conv2d(in_ch, out_ch, k, stride=stride, padding=padding, bias=False)
        self.weight_sum = nn.Conv2d(in_ch, out_ch, k, stride=stride, padding=padding, bias=False)
        nn.init.constant_(self.weight_sum.weight, 1.0)
        for p in self.weight_sum.parameters():
            p.requires_grad = False
        self.bias = nn.Parameter(torch.zeros(1, out_ch, 1, 1)) if bias else None
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, x, mask):
        masked_x   = x * mask
        raw_out    = self.conv(masked_x)
        norm_scale = torch.clamp(self.weight_sum(mask), min=1e-6)
        out = raw_out / norm_scale
        if self.bias is not None:
            out = out + self.bias
        out = self.bn(out)
        new_mask = (norm_scale > 0).float()
        return out, new_mask


class PartialConvMAE(nn.Module):
    """Partial-Conv UNet: (B,3,256,256) -> (B,1,256,256) dense path-loss."""
    def __init__(self):
        super().__init__()
        self.pconv1 = PartialConv2d(3,   32, 3, padding=1); self.act1 = nn.ReLU(inplace=True); self.pool1 = nn.MaxPool2d(2)
        self.pconv2 = PartialConv2d(32,  64, 3, padding=1); self.act2 = nn.ReLU(inplace=True); self.pool2 = nn.MaxPool2d(2)
        self.pconv3 = PartialConv2d(64, 128, 3, padding=1); self.act3 = nn.ReLU(inplace=True); self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up3  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec3 = nn.Sequential(nn.Conv2d(256+128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up2  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec2 = nn.Sequential(nn.Conv2d(128+64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up1  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec1 = nn.Sequential(nn.Conv2d(64+32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        mask    = (x[:, 2:3, :, :] > 0).float()
        mask_in = mask.expand(-1, 3, -1, -1)
        e1, m1 = self.pconv1(x, mask_in); e1 = self.act1(e1)
        e1p, m1p = self.pool1(e1), self.pool1(m1)
        e2, m2 = self.pconv2(e1p, m1p); e2 = self.act2(e2)
        e2p, m2p = self.pool2(e2), self.pool2(m2)
        e3, m3 = self.pconv3(e2p, m2p); e3 = self.act3(e3)
        e3p = self.pool3(e3)
        b  = self.bottleneck(e3p)
        d3 = self.dec3(torch.cat([self.up3(b),  e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)


# --- GNN (Vision GNN / ViG patch model) ------------------------------------
def knn_graph(x, k):
    """k-NN graph in feature space. x:(B,C,N,1) -> (B,N,k) neighbour indices."""
    B, C, N, _ = x.shape
    x_sq = x.squeeze(-1).permute(0, 2, 1)               # (B,N,C)
    dist = torch.cdist(x_sq, x_sq, p=2)                 # (B,N,N)
    _, idx = dist.topk(k + 1, dim=-1, largest=False)
    return idx[:, :, 1:]                                # drop self-loop


class DynConv2d(nn.Module):
    """EdgeConv-style dynamic graph conv on (B,C,N,1)."""
    def __init__(self, in_channels, out_channels, k=9):
        super().__init__()
        self.k    = k
        self.conv = nn.Conv2d(in_channels * 2, out_channels, 1)

    def forward(self, x):
        B, C, N, _ = x.shape
        idx       = knn_graph(x, self.k)
        x_flat    = x.squeeze(-1).permute(0, 2, 1)
        idx_exp   = idx.unsqueeze(-1).expand(-1, -1, -1, C)
        neighbors = torch.gather(
            x_flat.unsqueeze(2).expand(-1, -1, self.k, -1), 1, idx_exp)
        center = x_flat.unsqueeze(2).expand_as(neighbors)
        edge   = torch.cat([center, neighbors - center], dim=-1)
        agg    = edge.max(dim=2).values
        agg    = agg.permute(0, 2, 1).unsqueeze(-1)
        return self.conv(agg)


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        rand = keep + torch.rand(shape, dtype=x.dtype, device=x.device)
        rand.floor_()
        return x.div(keep) * rand


class GrapherModule(nn.Module):
    def __init__(self, in_channels, hidden_channels, k=9, drop_path=0.0):
        super().__init__()
        self.fc1 = nn.Sequential(nn.Conv2d(in_channels, in_channels, 1),
                                 nn.BatchNorm2d(in_channels))
        self.graph_conv = nn.Sequential(DynConv2d(in_channels, hidden_channels, k),
                                        nn.BatchNorm2d(hidden_channels), nn.GELU())
        self.fc2 = nn.Sequential(nn.Conv2d(hidden_channels, in_channels, 1),
                                 nn.BatchNorm2d(in_channels))
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x):
        shortcut = x
        x = self.fc1(x); x = self.graph_conv(x); x = self.fc2(x)
        return self.drop_path(x) + shortcut


class FFNModule(nn.Module):
    def __init__(self, in_channels, hidden_channels, drop_path=0.0):
        super().__init__()
        self.fc1 = nn.Sequential(nn.Conv2d(in_channels, hidden_channels, 1),
                                 nn.BatchNorm2d(hidden_channels), nn.GELU())
        self.fc2 = nn.Sequential(nn.Conv2d(hidden_channels, in_channels, 1),
                                 nn.BatchNorm2d(in_channels))
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x):
        shortcut = x
        x = self.fc1(x); x = self.fc2(x)
        return self.drop_path(x) + shortcut


class ViGBlock(nn.Module):
    def __init__(self, channels, k=9, drop_path=0.0):
        super().__init__()
        self.grapher = GrapherModule(channels, channels * 2, k, drop_path)
        self.ffn     = FFNModule(channels, channels * 4, drop_path)

    def forward(self, x):
        return self.ffn(self.grapher(x))


class PatchEmbed(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size=8):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, E, n_h, n_w = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x.transpose(1, 2).reshape(B, E, n_h, n_w)


class PatchUnembed(nn.Module):
    def __init__(self, embed_dim, out_channels, patch_size=8):
        super().__init__()
        self.proj = nn.ConvTranspose2d(embed_dim, out_channels, patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x)


class ViGPatchModel(nn.Module):
    """Vision GNN (ViG): (B,3,H,W) -> (B,1,H,W) dense path-loss."""
    def __init__(self, in_channels=3, out_channels=1, embed_dim=64,
                 num_blocks=4, patch_size=8, k=9, drop_path=0.1):
        super().__init__()
        self.patch_size  = patch_size
        self.patch_embed = PatchEmbed(in_channels, embed_dim, patch_size)
        dpr = [v.item() for v in torch.linspace(0, drop_path, num_blocks)]
        self.blocks = nn.ModuleList(
            [ViGBlock(embed_dim, k=k, drop_path=dpr[i]) for i in range(num_blocks)])
        self.norm          = nn.BatchNorm2d(embed_dim)
        self.patch_unembed = PatchUnembed(embed_dim, out_channels, patch_size)

    def _pad_to_patch(self, x):
        B, C, H, W = x.shape
        ph = (self.patch_size - H % self.patch_size) % self.patch_size
        pw = (self.patch_size - W % self.patch_size) % self.patch_size
        if ph > 0 or pw > 0:
            x = F.pad(x, (0, pw, 0, ph))
        return x, H, W

    def forward(self, x):
        x, orig_H, orig_W = self._pad_to_patch(x)
        B, C, H, W = x.shape
        x = self.patch_embed(x)
        _, E, n_h, n_w = x.shape
        x = x.reshape(B, E, n_h * n_w, 1)
        for blk in self.blocks:
            x = blk(x)
        x = x.reshape(B, E, n_h, n_w)
        x = self.norm(x)
        x = self.patch_unembed(x)
        return x[:, :, :orig_H, :orig_W]


# ── Checkpoint ───────────────────────────────
# Checkpoints are looked up by filename across candidate folders, so it works
# whether the .pth files sit next to the notebook or under Models/Trained_Models.
_CKPT_DIRS = ['.', 'Models/Trained_Models', 'models/trained-models']                # CHANGE DIRECTORY

def _find_ckpt(fname):
    for d in _CKPT_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return os.path.join(_CKPT_DIRS[0], fname)   # default path (for the warning)

MODELS_TO_COMPARE = ['WNet', 'CNN', 'PartialConvMAE', 'GNN']
MODEL_COLORS = {'WNet': '#DD8452', 'CNN': '#4C72B0',
                'PartialConvMAE': '#C44E52', 'GNN': '#8172B2'}

RECON_SPECS = {
    'WNet':           dict(file='Wnet.pt',                                          # CHANGE DIRECTORY
                           build=lambda: RadioWnet(inputs=3, phase='secondU')),
    'CNN':            dict(file='CNN_best.pth',                                     # CHANGE DIRECTORY
                           build=lambda: SpectrumCNN()),
    'PartialConvMAE': dict(file='PartialConvMAE_best.pth',                          # CHANGE DIRECTORY
                           build=lambda: PartialConvMAE()),
    'GNN':            dict(file='GNN_best.pth',                                     # CHANGE DIRECTORY
                           build=lambda: ViGPatchModel(in_channels=3, out_channels=1,
                                                       embed_dim=64, num_blocks=4,
                                                       patch_size=8, k=9, drop_path=0.1)),
}

def load_reconstructor(name):
    spec  = RECON_SPECS[name]
    model = spec['build']().to(device)
    path  = _find_ckpt(spec['file'])
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"  {name:16s} loaded {path}")
    else:
        print(f"  {name:16s} WARNING: {spec['file']} not found - UNTRAINED")
    model.eval()
    return model

@torch.no_grad()
def reconstruct(model, x_tensor):
    """Run any reconstruction model -> (256,256) numpy map clipped to [0,1].
    Handles RadioWnet's [out1, out2] list and single-tensor models."""
    out = model(x_tensor)
    if isinstance(out, (list, tuple)):
        out = out[-1]
    return np.clip(out.squeeze(0).squeeze(0).cpu().numpy(), 0.0, 1.0)

print('Reconstruction Models:', MODELS_TO_COMPARE)


Reconstruction Models: ['WNet', 'CNN', 'PartialConvMAE', 'GNN']


In [6]:
"""
── Free-space path-loss model ───────────────────────────────────────────
PL(d) = 50 + 30*log10(1 + d)  [dB], d in pixels from the emitter.
-
"""

_YY256, _XX256 = np.mgrid[0:256, 0:256].astype(np.float32)

def free_space_path_loss(tx_pos):
    d = np.hypot(_YY256 - tx_pos[0], _XX256 - tx_pos[1])
    return (50.0 + 30.0 * np.log10(1.0 + d)).astype(np.float32)


class RadioMapDataset:
    def __init__(self, zip_path, limit=None):
        self.data = []
        if not os.path.exists(zip_path):
            print(f"Error: {zip_path} not found.")
            return
        with zipfile.ZipFile(zip_path, 'r') as z:
            files = [f for f in z.namelist() if f.endswith('.parquet')]
            if limit:
                files = files[:limit]

            print(f"Loading {len(files)} samples from {zip_path}...")
            for f in files:
                with z.open(f) as pf:
                    df = pd.read_parquet(io.BytesIO(pf.read()))
                    row = df.iloc[0]
                    tx_mask = np.array(row['tx_origin']).reshape(256, 256)
                    tx_y, tx_x = np.unravel_index(np.argmax(tx_mask), (256, 256))
                    self.data.append({
                        'building_mask': np.zeros((256, 256), dtype=np.float32),    # omitted
                        'tx_pos': (tx_y, tx_x),
                        'path_loss': free_space_path_loss((tx_y, tx_x))
                    })
        print("Done loading.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = RadioMapDataset(os.path.join("Data", "train-001.zip"), limit=1000)                  # CHANGE DIRECTORY
val_dataset = RadioMapDataset(os.path.join("Data", "val.zip"), limit=100)                           # CHANGE DIRECTORY

Loading 1000 samples from Data/train-001.zip...
Done loading.
Loading 100 samples from Data/val.zip...
Done loading.


In [7]:
class RadioMapEnv(gym.Env):
    """
    Defines continuous 256x256 action space
    -
    Observation (4, 256, 256):
      ch0 - sparse path-loss samples (~obs_frac of disc cells, uniform random)
      ch1 - building mask (identically zero — kept so the architecture
            and replay format match ppo.ipynb exactly)
      ch2 - explored mask
      ch3 - agent position one-hot
    """

    def __init__(self, dataset, vis_radius=30, max_steps=250, step_size=5, obs_frac=0.15,
                 reach_dist=12.0, progress_weight=0.3, step_penalty=0.01,
                 pull_weight=0.15, pull_radius=15.0,
                 near_zone=25.0, near_penalty=0.05,
                 finish_bonus=50.0, shape_gamma=0.99):
        super().__init__()
        self.dataset    = dataset
        self.vis_radius = vis_radius
        self.max_steps  = max_steps
        self.step_size  = step_size
        self.obs_frac   = obs_frac
        self._geo_cache: dict = {}

        # ── Reward-shaping constants (named so they are tunable for A/B) ───
        # reach_dist 10->12 : a single radial step (step_size px) then lands
        #   inside the success disc from just outside it.
        # pull_* 0.5/25 -> 0.15/15 : the proximity potential no longer
        #   saturates the reward by ~15 px (which let the agent stall short).
        # near_penalty : extra living cost inside near_zone -> impatience to
        #   terminate rather than loiter near the emitter.
        # shape_gamma : DISCOUNTED potential shaping (gamma*phi' - phi); the
        #   old undiscounted form left standing still near the goal neutral.
        # finish_bonus 30->50 : termination dominates the near-goal value.
        self.reach_dist      = reach_dist
        self.progress_weight = progress_weight
        self.step_penalty    = step_penalty
        self.pull_weight     = pull_weight
        self.pull_radius     = pull_radius
        self.near_zone       = near_zone
        self.near_penalty    = near_penalty
        self.finish_bonus    = finish_bonus
        self.shape_gamma     = shape_gamma

        self.action_space      = spaces.Box(-1.0, 1.0, shape=(2,), dtype=np.float32)
        self.observation_space = spaces.Box(0.0, 1.0, shape=(4, 256, 256), dtype=np.float32)

        self.building_mask  = np.zeros((256, 256), dtype=np.float32)
        self.tx_pos         = (128, 128)
        self.path_loss      = np.zeros((256, 256), dtype=np.float32)
        self.norm_path_loss = np.zeros((256, 256), dtype=np.float32)
        self.geodesic_dist  = np.zeros((256, 256), dtype=np.float32)
        self.ugv_pos        = (0, 0)
        self.explored_mask  = np.zeros((256, 256), dtype=np.float32)
        self.observed_pl    = np.zeros((256, 256), dtype=np.float32)
        self.current_step   = 0
        self.prev_dist      = 0.0
        self.prev_euc       = 0.0

    # ── Geodesic precomputation ───────────────────────────────────────────
    def _compute_geodesic_dist(self): # fancy word for shortest path.
        """Free space geodisc"""
        ty, tx = self.tx_pos
        yy, xx = np.ogrid[:256, :256]
        return np.hypot(yy - ty, xx - tx).astype(np.float32)

    # ── Episode reset ─────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if not self.dataset.data:
            return np.zeros((4, 256, 256), dtype=np.float32), {}

        idx                 = np.random.randint(len(self.dataset))
        sample              = self.dataset[idx]
        self.building_mask  = sample['building_mask'].astype(np.float32)
        self.tx_pos         = sample['tx_pos']
        self.path_loss      = sample['path_loss'].astype(np.float32)
        self.norm_path_loss = np.clip((self.path_loss - 50.0) / 100.0, 0.0, 1.0)

        if idx not in self._geo_cache:
            self._geo_cache[idx] = self._compute_geodesic_dist()
        self.geodesic_dist = self._geo_cache[idx]

        free = np.argwhere(self.building_mask == 0)   # mask is all zero: every cell
        max_start_dist = (options or {}).get('max_start_dist', None)
        if max_start_dist is not None:
            d = np.hypot(free[:, 0] - self.tx_pos[0], free[:, 1] - self.tx_pos[1])
            cand = free[d <= max_start_dist]
            if len(cand) >= 5:
                free = cand

        self.ugv_pos       = tuple(free[np.random.randint(len(free))])
        self.explored_mask = np.zeros((256, 256), dtype=np.float32)
        self.observed_pl   = np.zeros((256, 256), dtype=np.float32)
        self.current_step  = 0
        self.prev_dist     = float(self.geodesic_dist[self.ugv_pos])
        self.prev_euc      = self._dist_to_tx(self.ugv_pos)

        self._update_visibility()
        return self._get_obs(), {}

    # ── Helpers ───────────────────────────────────────────────────────────
    def _dist_to_tx(self, pos):
        return float(np.hypot(pos[0] - self.tx_pos[0], pos[1] - self.tx_pos[1]))

    def _update_visibility(self):
        y, x   = self.ugv_pos
        yg, xg = np.ogrid[:256, :256]
        vis    = ((yg - y) ** 2 + (xg - x) ** 2) <= self.vis_radius ** 2
        self.explored_mask = np.maximum(self.explored_mask, vis.astype(np.float32))
        # Free space: every cell in the visibility disc is measurable.
        candidate_idx = np.argwhere(vis)
        if len(candidate_idx) > 0:
            n_sample = max(1, int(round(len(candidate_idx) * self.obs_frac)))
            chosen   = candidate_idx[
                np.random.choice(len(candidate_idx), n_sample, replace=False)
            ]
            cy, cx = chosen[:, 0], chosen[:, 1]
            self.observed_pl[cy, cx] = np.maximum(
                self.observed_pl[cy, cx], self.norm_path_loss[cy, cx]
            )

    def _get_obs(self):
        pos               = np.zeros((256, 256), dtype=np.float32)
        pos[self.ugv_pos] = 1.0
        return np.stack([self.observed_pl, self.building_mask,
                         self.explored_mask, pos], axis=0)

    # ── Movement ──────────────────────────────────────────────────────────
    def _trace_step(self, dy_f, dx_f):
        """Free space """
        y0, x0 = self.ugv_pos
        ny = int(np.clip(round(y0 + dy_f), 0, 255))
        nx = int(np.clip(round(x0 + dx_f), 0, 255))
        return ny, nx

    # ── Environment step ──────────────────────────────────────────────────
    def step(self, action):
        self.current_step += 1
        # Clip here rather than in the agent: the policy samples an unbounded
        # Gaussian and its stored log-probs must describe the raw sample, so
        # the action bound is enforced at the environment boundary instead.
        dy = float(np.clip(action[0], -1.0, 1.0)) * self.step_size
        dx = float(np.clip(action[1], -1.0, 1.0)) * self.step_size

        new_y, new_x = self._trace_step(dy, dx)
        self.ugv_pos = (new_y, new_x)

        self._update_visibility()

        geo_new = float(self.geodesic_dist[self.ugv_pos])
        euc_new = self._dist_to_tx(self.ugv_pos)

        # Long-range: Euclidean progress shaping. In free space the straight
        # line IS the geodesic — there is nothing to route around.
        reward = (self.prev_dist - geo_new) * self.progress_weight - self.step_penalty

        # Impatience: an extra living cost inside the near-zone so loitering
        # near the emitter without terminating is strictly costly.
        if euc_new < self.near_zone:
            reward -= self.near_penalty

        # Short-range: DISCOUNTED potential-based pull-in within pull_radius.
        #   phi(s) = pull_weight * max(0, pull_radius - euc)
        #   F      = shape_gamma * phi(s') - phi(s)
        # Discounting makes standing still near the goal mildly negative (the
        # old phi'-phi left it neutral), and the smaller weight/radius stop the
        # potential from saturating the reward ~15 px out — the saturation that
        # let the agent stall short of the emitter.
        phi_new  = self.pull_weight * max(0.0, self.pull_radius - euc_new)
        phi_prev = self.pull_weight * max(0.0, self.pull_radius - self.prev_euc)
        reward  += self.shape_gamma * phi_new - phi_prev

        terminated = euc_new < self.reach_dist
        truncated  = self.current_step >= self.max_steps
        if terminated:
            reward += self.finish_bonus  # successful path
        self.prev_dist = geo_new
        self.prev_euc  = euc_new

        info = {
            'dist_to_tx': euc_new,
            'coverage'  : float(self.explored_mask.mean()),
            'signal'    : float(self.norm_path_loss[self.ugv_pos]),
        }
        return self._get_obs(), float(reward), terminated, truncated, info

    @property
    def agent_row(self): return self.ugv_pos[0]
    @property
    def agent_col(self): return self.ugv_pos[1]


In [8]:
import torch.nn.functional as F
from torch.distributions import Normal

# ── State from the reconstruction ─────────────────────────────────────────
#
#   state = [dir_y, dir_x, dist_norm]
#     dir       : unit vector from the UGV to argmin(recon) (estimated emitter)
#     dist_norm : that distance / map-diagonal, so the agent can slow down and
#                 stop at the goal instead of overshooting it.

DIAG_PX = float(np.hypot(255.0, 255.0))

def recon_state(recon, ugv_pos):
    gy, gx = np.unravel_index(int(np.argmin(recon)), recon.shape)
    dy   = float(gy - ugv_pos[0])
    dx   = float(gx - ugv_pos[1])
    dist = float(np.hypot(dy, dx))
    inv  = 1.0 / (dist + 1e-6)
    return np.array([dy * inv, dx * inv, dist / DIAG_PX], dtype=np.float32)


# ── Actor-Critic: small MLP over the 3-D state ────────────────────────────
class ActorCritic(nn.Module):
    def __init__(self, obs_dim=3, action_dim=2):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64),      nn.Tanh(),
        )
        self.actor_mean    = nn.Linear(64, action_dim)   # raw, pre-squash mean
        self.actor_log_std = nn.Parameter(torch.full((action_dim,), -0.5))
        self.critic        = nn.Linear(64, 1)
        nn.init.orthogonal_(self.actor_mean.weight, gain=0.01)
        nn.init.zeros_(self.actor_mean.bias)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x):
        h = self.body(x)
        return self.actor_mean(h), self.critic(h)

    def get_dist_and_value(self, x):
        mean, val = self.forward(x)
        std = self.actor_log_std.exp().clamp(1e-3, 2.0)
        return Normal(mean, std), val


# ── PPO with GAE (tanh-squashed Gaussian; ratio taken in latent u-space) ───
class PPOAgent:
    def __init__(self, obs_dim=3, action_dim=2, lr=3e-4, gamma=0.99, lam=0.95,
                 eps_clip=0.2, k_epochs=10, minibatch=256, ent_coef=0.0):
        self.policy     = ActorCritic(obs_dim, action_dim).to(device)
        self.policy_old = ActorCritic(obs_dim, action_dim).to(device)
        self.policy_old.load_state_dict(self.policy.state_dict())
        self.optim      = optim.Adam(self.policy.parameters(), lr=lr, eps=1e-5)

        self.gamma, self.lam = gamma, lam
        self.eps_clip        = eps_clip
        self.k_epochs        = k_epochs
        self.minibatch       = minibatch
        self.ent_coef        = ent_coef

    @torch.no_grad()
    def act(self, state):
        # 2 hidden layers size 64 w/ tanh activations
        t = torch.FloatTensor(state).unsqueeze(0).to(device)
        dist, val = self.policy_old.get_dist_and_value(t)
        u  = dist.sample()
        a  = torch.tanh(u)
        lp = dist.log_prob(u).sum(-1)
        return u.cpu().numpy()[0], a.cpu().numpy()[0], lp.item(), val.item()

    def _gae(self, rewards, dones, values, last_val=0.0):
        n   = len(rewards)
        adv = np.zeros(n, dtype=np.float32)
        gae, nxt = 0.0, last_val
        for i in reversed(range(n)):
            d      = float(dones[i])
            delta  = rewards[i] + self.gamma * nxt * (1.0 - d) - values[i]
            gae    = delta + self.gamma * self.lam * (1.0 - d) * gae
            adv[i] = gae
            nxt    = values[i]
        return adv, adv + np.array(values, dtype=np.float32)

    def update(self, buf, last_val=0.0):
        adv, ret = self._gae(buf['rewards'], buf['dones'], buf['values'], last_val)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        ts   = torch.FloatTensor(np.stack(buf['obs'])).to(device)
        ta   = torch.FloatTensor(np.stack(buf['actions'])).to(device)
        tlp  = torch.FloatTensor(np.array(buf['log_probs'], np.float32)).to(device)
        tadv = torch.FloatTensor(adv).to(device)
        tret = torch.FloatTensor(ret).to(device)

        n    = len(buf['rewards'])
        logs = {'pi': [], 'vf': [], 'ent': []}

        for _ in range(self.k_epochs):
            perm = torch.randperm(n)
            for s in range(0, n, self.minibatch):
                idx       = perm[s: s + self.minibatch]
                dist, val = self.policy.get_dist_and_value(ts[idx])
                nlp       = dist.log_prob(ta[idx]).sum(-1)
                ent       = dist.entropy().sum(-1)
                ratio     = (nlp - tlp[idx]).exp()
                pg        = -torch.min(
                    ratio * tadv[idx],
                    ratio.clamp(1 - self.eps_clip, 1 + self.eps_clip) * tadv[idx]
                ).mean()
                vl        = 0.5 * (val.squeeze(-1) - tret[idx]).pow(2).mean()
                el        = -self.ent_coef * ent.mean()
                loss      = pg + vl + el
                self.optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.optim.step()
                logs['pi'].append(pg.item())
                logs['vf'].append(vl.item())
                logs['ent'].append(ent.mean().item())

        self.policy_old.load_state_dict(self.policy.state_dict())
        return {k: float(np.mean(v)) for k, v in logs.items()}


In [10]:
# ── Hyperparameters ───────────────────────────────────────────────────────
MAX_EPISODES    = 800
BUFFER_SIZE     = 2048
LOG_INTERVAL    = 20
CURRICULUM_RAMP = 0.75
REACH_DIST      = 12.0     # keep in sync with RadioMapEnv.reach_dist


def _flush(buf, agent, hist, min_size=64):
    if len(buf['rewards']) < min_size:
        return buf
    m = agent.update(buf, last_val=buf['values'][-1])
    hist['pi'].append(m['pi']); hist['vf'].append(m['vf']); hist['ent'].append(m['ent'])
    return {k: [] for k in buf}


def train_ppo(name, recon_model, live_plot=True):
    """Train one PPO agent whose state comes from `recon_model`'s argmin.
    Returns (agent, history)."""
    env   = RadioMapEnv(train_dataset, step_size=5, vis_radius=35, obs_frac=0.15)
    agent = PPOAgent(obs_dim=3, action_dim=2)
    buf   = {k: [] for k in ['obs', 'actions', 'log_probs', 'rewards', 'dones', 'values']}
    hist  = {k: [] for k in ['ep_reward', 'ep_len', 'success', 'dist_final', 'pi', 'vf', 'ent']}
    total_steps = 0

    for ep in range(1, MAX_EPISODES + 1):
        t_frac         = min(ep / (MAX_EPISODES * CURRICULUM_RAMP), 1.0)
        max_start_dist = 50.0 + 170.0 * t_frac
        obs, _ = env.reset(options={'max_start_dist': max_start_dist})
        ep_rew = 0.0; ep_len = 0; reached = False

        for t in range(env.max_steps):
            wnet_in = build_wnet_input(env, env.observed_pl)
            recon   = reconstruct(recon_model,
                                  torch.FloatTensor(wnet_in).unsqueeze(0).to(device))
            state   = recon_state(recon, env.ugv_pos)

            action_u, action_env, lp, val = agent.act(state)
            buf['obs'].append(state)
            buf['actions'].append(action_u.copy())
            buf['log_probs'].append(lp)
            buf['values'].append(val)

            obs, rew, term, trunc, info = env.step(action_env)
            done    = term or trunc
            buf_rew = rew + (agent.gamma * val if (trunc and not term) else 0.0)
            buf['rewards'].append(buf_rew)
            buf['dones'].append(float(done))

            ep_rew += rew; ep_len += 1; total_steps += 1
            if info['dist_to_tx'] < REACH_DIST:
                reached = True
            if len(buf['rewards']) >= BUFFER_SIZE:
                buf = _flush(buf, agent, hist)
            if done:
                break

        hist['ep_reward'].append(ep_rew)
        hist['ep_len'].append(ep_len)
        hist['success'].append(float(reached))
        hist['dist_final'].append(info['dist_to_tx'])

        if live_plot and ep % LOG_INTERVAL == 0:
            sr20 = np.mean(hist['success'][-20:])
            df20 = np.mean(hist['dist_final'][-20:])
            clear_output(wait=True)
            c = MODEL_COLORS[name]
            fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
            ax[0].plot(pd.Series(hist['ep_reward']).rolling(20).mean(), c=c)
            ax[0].set(title='Episode Reward (20-ep MA)', xlabel='Episode')
            ax[1].plot(pd.Series(hist['success']).rolling(20).mean(), c=c)
            ax[1].set(title='Success Rate (20-ep MA)', xlabel='Episode', ylim=(0, 1))
            ax[2].plot(pd.Series(hist['dist_final']).rolling(20).mean(), c=c)
            ax[2].axhline(REACH_DIST, c='green', ls='--', lw=1)
            ax[2].set(title='Final Distance (px, 20-ep MA)', xlabel='Episode')
            fig.suptitle(f'Training {name}   |   ep {ep}/{MAX_EPISODES}   |   '
                         f'success(20) {sr20:.0%}   |   dist {df20:.0f}px',
                         fontweight='bold')
            plt.tight_layout(); plt.show()

    buf = _flush(buf, agent, hist)
    torch.save(agent.policy.state_dict(), f'ppo_freespace_{name}.pth')
    print(f'[{name}] done | final success (last 50): '
          f'{np.mean(hist["success"][-50:]):.1%} | saved ppo_freespace_{name}.pth')
    return agent, hist


# ── Train one PPO per reconstruction model ────────────────────────────────
print('Loading reconstruction models...')
recon_models = {name: load_reconstructor(name) for name in MODELS_TO_COMPARE}

agents, histories = {}, {}
for name in MODELS_TO_COMPARE:
    print(f'\n=== Training PPO with {name} reconstruction ===')
    agents[name], histories[name] = train_ppo(name, recon_models[name])

# ── Clean overlaid training-curve comparison ──────────────────────────────
clear_output(wait=True)
print('All training complete.\n')
for name in MODELS_TO_COMPARE:
    print(f'  {name:16s} final success(50) {np.mean(histories[name]["success"][-50:]):.0%}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name in MODELS_TO_COMPARE:
    h, c = histories[name], MODEL_COLORS[name]
    axes[0].plot(pd.Series(h['ep_reward']).rolling(20).mean(),  c=c, label=name)
    axes[1].plot(pd.Series(h['success']).rolling(20).mean(),    c=c, label=name)
    axes[2].plot(pd.Series(h['dist_final']).rolling(20).mean(), c=c, label=name)
axes[0].set(title='Episode Reward (20-ep MA)', xlabel='Episode', ylabel='Reward')
axes[1].set(title='Success Rate (20-ep MA)',   xlabel='Episode', ylabel='Rate', ylim=(0, 1))
axes[2].axhline(REACH_DIST, c='gray', ls='--', lw=1)
axes[2].set(title='Final Distance (20-ep MA)', xlabel='Episode', ylabel='Pixels')
for ax in axes:
    ax.legend(fontsize=9)
fig.suptitle('PPO Training - Reconstruction-Model Ablation', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_training_ablation_freespace.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_training_ablation_freespace.pdf')


Loading reconstruction models...
  WNet             WARNING: Wnet.pt not found - UNTRAINED
  CNN              loaded Models/Trained_Models/CNN_best.pth
  PartialConvMAE   loaded Models/Trained_Models/PartialConvMAE_best.pth


RuntimeError: Error(s) in loading state_dict for ViGPatchModel:
	Missing key(s) in state_dict: "patch_embed.proj.weight", "patch_embed.proj.bias", "patch_embed.norm.weight", "patch_embed.norm.bias", "blocks.0.grapher.graph_conv.0.conv.weight", "blocks.0.grapher.graph_conv.0.conv.bias", "blocks.0.grapher.graph_conv.1.weight", "blocks.0.grapher.graph_conv.1.bias", "blocks.0.grapher.graph_conv.1.running_mean", "blocks.0.grapher.graph_conv.1.running_var", "blocks.0.ffn.fc1.0.weight", "blocks.0.ffn.fc1.0.bias", "blocks.0.ffn.fc1.1.weight", "blocks.0.ffn.fc1.1.bias", "blocks.0.ffn.fc1.1.running_mean", "blocks.0.ffn.fc1.1.running_var", "blocks.0.ffn.fc2.0.weight", "blocks.0.ffn.fc2.0.bias", "blocks.0.ffn.fc2.1.weight", "blocks.0.ffn.fc2.1.bias", "blocks.0.ffn.fc2.1.running_mean", "blocks.0.ffn.fc2.1.running_var", "blocks.1.grapher.graph_conv.0.conv.weight", "blocks.1.grapher.graph_conv.0.conv.bias", "blocks.1.grapher.graph_conv.1.weight", "blocks.1.grapher.graph_conv.1.bias", "blocks.1.graphe

In [ ]:
# ── Style ──────────────────────────────────────────────────────────────────
STYLE = {
    'figure.dpi': 130, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans',
    'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'lines.linewidth': 2.0,
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
}
plt.rcParams.update(STYLE)
PALETTE = dict(path='#00d4ff', emitter='#FFD700', start='#44ff88', end='#ff4444')


# ── Deterministic rollout for one (agent, recon_model) pair ────────────────
def evaluate_policy(env, agent, recon_model, n_episodes=12, seed0=None):
    """Run n_episodes deterministic rollouts. If seed0 is set, scenario i uses
    seed0+i so every model is evaluated on the SAME start/emitter layouts."""
    results = []
    for i in range(n_episodes):
        if seed0 is not None:
            np.random.seed(seed0 + i)
        env.reset()
        path = [env.ugv_pos]
        for step in range(env.max_steps):
            wnet_in = build_wnet_input(env, env.observed_pl)
            recon   = reconstruct(recon_model,
                                  torch.FloatTensor(wnet_in).unsqueeze(0).to(device))
            state   = recon_state(recon, env.ugv_pos)
            t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                mean, _ = agent.policy(t)
                action  = torch.tanh(mean).cpu().numpy()[0]
            _, _, term, trunc, info = env.step(action)
            path.append(env.ugv_pos)
            if term or trunc:
                break
        pa = np.array(path)
        results.append(dict(success=info['dist_to_tx'] < 12.0,
                            final_dist=info['dist_to_tx'], n_steps=len(pa),
                            coverage=info['coverage'], path=pa,
                            tx_pos=env.tx_pos, path_loss=env.path_loss.copy()))
    return results


# ── Figure 1: summary metric bars ──────────────────────────────────────────
def plot_summary(eval_all, save_path='fig_summary_ablation_freespace.pdf'):
    names = list(eval_all)
    cols  = [MODEL_COLORS[n] for n in names]
    sr = [100 * np.mean([r['success']    for r in eval_all[n]]) for n in names]
    fd = [      np.mean([r['final_dist'] for r in eval_all[n]]) for n in names]
    ns = [      np.mean([r['n_steps']    for r in eval_all[n]]) for n in names]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
    for a, vals, title in zip(ax, [sr, fd, ns],
                              ['Success Rate (%)', 'Mean Final Distance (px)',
                               'Mean Steps per Episode']):
        bars = a.bar(names, vals, color=cols, edgecolor='white')
        a.set_title(title)
        a.tick_params(axis='x', rotation=12)
        for b, v in zip(bars, vals):
            a.text(b.get_x() + b.get_width() / 2, v, f'{v:.0f}',
                   ha='center', va='bottom', fontsize=9)
    ax[0].set_ylim(0, 100)
    ax[1].axhline(12, c='green', ls='--', lw=1.2)
    fig.suptitle('PPO Navigation - Reconstruction-Model Comparison', fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight'); plt.show()
    print(f'Saved {save_path}')


# ── Figure 2: trajectory grid (rows = model, cols = matched scenarios) ──────
def plot_trajectory_grid(eval_all, n_per=5,
                         save_path='fig_trajectories_ablation_freespace.pdf'):
    names = list(eval_all)
    nrows, ncols = len(names), n_per
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.5 * ncols, 2.7 * nrows),
                             squeeze=False)
    for ri, name in enumerate(names):
        res = eval_all[name]
        for ci in range(ncols):
            ax = axes[ri][ci]
            if ci >= len(res):
                ax.axis('off'); continue
            r  = res[ci]; pa = r['path']; tx = r['tx_pos']
            ax.imshow(r['path_loss'], cmap='viridis', origin='upper',
                      interpolation='nearest')
            ax.plot(pa[:, 1], pa[:, 0], color=PALETTE['path'], lw=1.2, zorder=3)
            ax.scatter(pa[0, 1], pa[0, 0], marker='o', c=PALETTE['start'], s=20,
                       zorder=5, edgecolors='#222', linewidths=0.4)
            ax.scatter(tx[1], tx[0], marker='*', c=PALETTE['emitter'], s=80,
                       zorder=6, edgecolors='#222', linewidths=0.4)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"d={r['final_dist']:.0f}px  n={r['n_steps']}", fontsize=7,
                         color='seagreen' if r['success'] else 'tomato', pad=2)
            if ci == 0:
                ax.set_ylabel(name, fontsize=11, fontweight='bold',
                              color=MODEL_COLORS[name])
    fig.suptitle('Validation Trajectories  (rows = recon model, columns = matched scenarios)\n'
                 '* emitter   o start   - path',
                 fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, bbox_inches='tight'); plt.show()
    print(f'Saved {save_path}')


# ── Figure 3: reconstruction & path evolution (rows = step, cols = model) ──
def plot_recon_evolution(recon_models, agents, env, scenario_seed=123,
                         snap_every=5, n_rows=6, max_start_dist=200,
                         save_path='fig_recon_comparison_ablation_freespace.pdf'):
    """Adjust scenario_seed for different ground truths"""
    names = list(recon_models)
    opts  = {'max_start_dist': max_start_dist}

    # Roll out every model on the SAME scenario, storing strided snapshots.
    rollouts = {}
    for name in names:
        np.random.seed(scenario_seed)
        env.reset(options=opts)
        path, snaps = [env.ugv_pos], {}
        for step in range(env.max_steps):
            wnet_in = build_wnet_input(env, env.observed_pl)
            recon   = reconstruct(recon_models[name],
                                  torch.FloatTensor(wnet_in).unsqueeze(0).to(device))
            if step % snap_every == 0:
                snaps[step] = dict(recon=recon.copy(), path=list(path), pos=env.ugv_pos)
            state = recon_state(recon, env.ugv_pos)
            t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                mean, _ = agents[name].policy(t)
                action  = torch.tanh(mean).cpu().numpy()[0]
            _, _, term, trunc, _ = env.step(action)
            path.append(env.ugv_pos)
            if term or trunc:
                snaps[step] = dict(recon=recon.copy(), path=list(path),
                                   pos=env.ugv_pos, done=bool(term))
                break
        rollouts[name] = snaps

    # Ground-truth field for this scenario (re-derive deterministically).
    np.random.seed(scenario_seed); env.reset(options=opts)
    gt, tx = env.path_loss.copy(), env.tx_pos

    # Row step values span 0 .. longest rollout so the last row shows the end.
    max_last     = max(max(s) for s in rollouts.values())
    target_steps = sorted(set(int(round(v)) for v in np.linspace(0, max_last, n_rows)))
    nrows, ncols = len(target_steps), 1 + len(names)

    fig, axes = plt.subplots(nrows, ncols, figsize=(2.7 * ncols, 2.7 * nrows),
                             squeeze=False)
    for ri, ts in enumerate(target_steps):
        ax = axes[ri][0]
        ax.imshow(gt, cmap='viridis', origin='upper', interpolation='nearest')
        ax.scatter(tx[1], tx[0], marker='*', c=PALETTE['emitter'], s=80,
                   edgecolors='#222', linewidths=0.4, zorder=5)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylabel(f'step {ts}', fontsize=10, fontweight='bold')
        if ri == 0:
            ax.set_title('Ground Truth', fontsize=10)
        for mi, name in enumerate(names):
            snaps  = rollouts[name]
            key    = max(s for s in snaps if s <= ts)      # freeze after termination
            snap   = snaps[key]
            pa     = np.array(snap['path'])
            gy, gx = np.unravel_index(int(np.argmin(snap['recon'])), snap['recon'].shape)
            ax = axes[ri][mi + 1]
            ax.imshow(snap['recon'], cmap='viridis', origin='upper', interpolation='nearest')
            if len(pa) > 1:
                ax.plot(pa[:, 1], pa[:, 0], color=PALETTE['path'], lw=1.3, zorder=3)
            ax.scatter(snap['pos'][1], snap['pos'][0], marker='o', c=PALETTE['start'],
                       s=22, zorder=5, edgecolors='#222', linewidths=0.4)
            ax.scatter(gx, gy, marker='x', c='red', s=45, linewidths=1.4, zorder=6)
            ax.scatter(tx[1], tx[0], marker='*', c=PALETTE['emitter'], s=70,
                       edgecolors='#222', linewidths=0.4, zorder=7)
            ax.set_xticks([]); ax.set_yticks([])
            if ri == 0:
                ax.set_title(name, fontsize=10, color=MODEL_COLORS[name])
            if snap.get('done'):
                ax.text(0.5, 0.05, 'reached', transform=ax.transAxes, ha='center',
                        fontsize=7, color='seagreen', fontweight='bold')
    fig.suptitle('Reconstruction & Path Evolution  (rows = step, cols = model;  '
                 '* emitter   x argmin   o agent   - path)', fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(save_path, bbox_inches='tight'); plt.show()
    print(f'Saved {save_path}')


# ── Load models/agents ─────────────────
# cancel run and run next block to visualize previously trained ppos
val_env = RadioMapEnv(val_dataset, step_size=5, vis_radius=35, obs_frac=0.15)

if 'recon_models' not in dir():
    recon_models = {}
for n in MODELS_TO_COMPARE:
    if n not in recon_models:
        recon_models[n] = load_reconstructor(n)

if 'agents' not in dir():
    agents = {}
for n in MODELS_TO_COMPARE:
    if n not in agents:
        a, p = PPOAgent(obs_dim=3, action_dim=2), f'ppo_freespace_{n}.pth'
        if os.path.exists(p):
            a.policy.load_state_dict(torch.load(p, map_location=device))
            print(f'  loaded {p}')
        else:
            print(f'  WARNING: {p} not found - {n} agent is UNTRAINED')
        a.policy.eval()
        agents[n] = a

# ── Evaluate every model on the SAME validation scenarios ──────────────────
N_VAL = 12
print(f'\nEvaluating {len(MODELS_TO_COMPARE)} models on {N_VAL} matched scenarios...')
eval_all = {n: evaluate_policy(val_env, agents[n], recon_models[n],
                               n_episodes=N_VAL, seed0=1000)
            for n in MODELS_TO_COMPARE}
for n in MODELS_TO_COMPARE:
    sr = np.mean([r['success'] for r in eval_all[n]])
    fd = np.mean([r['final_dist'] for r in eval_all[n]])
    print(f'  {n:16s} success {sr:.0%} | mean final dist {fd:.1f}px')

plot_summary(eval_all)
plot_trajectory_grid(eval_all, n_per=5)
plot_recon_evolution(recon_models, agents, val_env, snap_every=5, n_rows=20)
